# New regime model

A regime model is a function of one instrument's bars that returns state probabilities per bar (rows sum to 1),
using only data up to each bar. It has to show three things, in this order:

1. **It can find regimes that exist**: the synthetic bench, against simple detectors.
2. **Its state predicts which strategy earns more next**: the regime card on real data, with a timing placebo.
3. **Switching on it beats not switching**: the overlay against its static blend and exposure-matched benchmark.

**Hypothesis.** _Which feature separates which states, and why should the states matter for returns?_

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quantlab.lab import Lab
from quantlab.signals import library as lib
from quantlab.reporting import charts

charts.apply_style()
pd.set_option("display.width", 200)

from quantlab.regimes.models import PercentileRegime, VolatilityRegime, FeatureBlendRegime, labels
from quantlab.regimes.simulate import RegimeProcess, simulate_paths
from quantlab.regimes.bench import bench
from quantlab.validation.regime_card import regime_card

## The model

Any feature of bars becomes a two-state model through its walk-forward percentile (`PercentileRegime`). Or write a class with `states`, `lag` and `__call__(bars)`.

In [ ]:
def trend_quality(bars):
    returns = np.log(bars["close"]).diff()
    return returns.rolling(60).sum().abs() / returns.abs().rolling(60).sum()   # efficiency over 60 days

model = PercentileRegime(trend_quality, state_names=("RANGE", "TREND"), confirm=3, name="trend quality 60")

## 1. The synthetic bench

In [ ]:
paths = simulate_paths(RegimeProcess(), years=15, seeds=range(4))
two = {"TREND": "TREND", "RANGE": "RANGE"}
bench({"mine": (model, two), "efficiency ratio": (PercentileRegime("er"), two),
       "H14 blend": (FeatureBlendRegime(), {**two, "TRANSITION": None})}, paths)[
    ["coverage", "balanced accuracy", "median lag (bars)", "false switches a year"]].round(3)

## 2. Regime cards on real data

Per market: does the state at the close predict whether trend beats mean reversion over the next 20 days?

In [ ]:
lab = Lab(universe="equity_index_core", start="1999", show=False)
trend, reversion = lab.backtest("trend"), lab.backtest("reversion")
cards = {}
for market in lab.instruments:
    probabilities = model(lab.panel.bars(market))
    cards[market] = regime_card(probabilities, trend.net[market], reversion.net[market], state_a="TREND", state_b="RANGE",
                                name=f"{model.name} on {market}", bars=lab.panel.bars(market), ledger=False)
pd.DataFrame({m: {"verdict": c.verdict, "edge": c.details["edge"], "placebo p": c.details["placebo_p"], "switches a year": c.details["switches"]}
              for m, c in cards.items()}).T

Several markets are several tests. Count them: one market passing among seven is what luck gives. `ledger=False`
keeps these exploratory; in a confirmatory run (pre-registered) set `ledger=True, family="H<n>"`.

## 3. The overlay

In [ ]:
overlay = lab.combine("trend", "reversion", overlay=model, show=True,
                      state_weights={"TREND": {"trend": 1.0}, "RANGE": {"reversion": 1.0}})